# XAI on Colab: Grad-CAM and Integrated Gradients

**Before running:** Runtime → Change runtime type → **T4 GPU**.

What this notebook does:
- **Sections 5–8:** Grad-CAM **and** IG on the 860 test tumor images, for the representative seed of each model (about 30–45 min per model). Used for RQ2 and RQ3.
- **Section 9:** Grad-CAM only for the other 8 runs (a few minutes each). Used for RQ4.

Results are written to Google Drive as they go. **If the session disconnects:** run cells 1–4 again, then rerun the cell that was interrupted. Images already done are skipped automatically.

## 1. Connect Google Drive
The trained models are read from Drive, and the results are saved there.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Get the code and data
Clones the repo the first time; pulls the latest code if it's already there.

In [2]:
import os

REPO = "/content/COMP-6936_Project"
if os.path.exists(REPO):
    %cd {REPO}
    !git pull
else:
    !git clone https://github.com/labib-islam/COMP-6936_Project.git {REPO}
    %cd {REPO}

!git log --oneline -1

Cloning into '/content/COMP-6936_Project'...
remote: Enumerating objects: 10908, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 10908 (delta 73), reused 142 (delta 56), pack-reused 10749 (from 1)
Receiving objects: 100% (10908/10908), 140.91 MiB | 34.58 MiB/s, done.
Resolving deltas: 100% (77/77), done.
Updating files: 100% (15663/15663), done.
/content/COMP-6936_Project
6da089b (HEAD -> main, origin/main, origin/HEAD) Add XAI scoring and Colab explain notebook


## 3. Check the GPU
Should print `True` and `Tesla T4`.

In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

name, memory.total [MiB]
Tesla T4, 15360 MiB
2.14.1+cu130 True Tesla T4


## 4. Settings
`RUNS` is the Drive folder with the 12 trained runs (each with `best.pt`). `OUT` is where the XAI results go.

The representative seeds come from `results/test_summary.csv`: the seed whose val F1 is closest to the model's mean.

In [4]:
DRIVE = "/content/drive/MyDrive/COMP6936 - Advanced Machine Learning"
RUNS = f"{DRIVE}/runs"
OUT = f"{DRIVE}/xai"

REP_SEEDS = {"densenet121": 42, "convnext_tiny": 43, "resnet50": 43, "efficientnet_v2_s": 44}
ALL_SEEDS = [42, 43, 44]

os.makedirs(OUT, exist_ok=True)

# check that all 12 checkpoints are on Drive
for model in REP_SEEDS:
    for seed in ALL_SEEDS:
        path = f"{RUNS}/{model}_lr0.0001_seed{seed}/best.pt"
        print("ok     " if os.path.exists(path) else "MISSING", path)

ok      /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/runs/densenet121_lr0.0001_seed42/best.pt
ok      /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/runs/densenet121_lr0.0001_seed43/best.pt
ok      /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/runs/densenet121_lr0.0001_seed44/best.pt
ok      /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/runs/convnext_tiny_lr0.0001_seed42/best.pt
ok      /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/runs/convnext_tiny_lr0.0001_seed43/best.pt
ok      /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/runs/convnext_tiny_lr0.0001_seed44/best.pt
ok      /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/runs/resnet50_lr0.0001_seed42/best.pt
ok      /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/runs/resnet50_lr0.0001_seed43/best.pt
ok      /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/runs/resnet50_lr0.0001_seed44/best.pt
ok      /content/drive

## 5. Quick checks
The score unit tests, then a 3-image run to a temporary folder (not Drive). Both take under a minute.

In [5]:
!python -m src.xai.scores
!python -u -m src.xai.explain --model densenet121 --seed 42 --runs "{RUNS}" --out /content/xai_test --limit 3

All score tests passed
Explaining densenet121_lr0.0001_seed42 on cuda, methods ['gradcam', 'ig']

gradcam: 0 images already done, 3 to go -> /content/xai_test/densenet121_lr0.0001_seed42/gradcam_scores.csv
  3/3 images, 0.87s per image, about 0 min left

ig: 0 images already done, 3 to go -> /content/xai_test/densenet121_lr0.0001_seed42/ig_scores.csv
  3/3 images, 2.20s per image, about 0 min left

Summary (predicted class, all images so far):
  gradcam    n    3 | pointing 0.333 | energy 0.171 | energy ratio 4.7
  ig_raw     n    3 | pointing 0.667 | energy 0.239 | energy ratio 7.7
  ig_smooth  n    3 | pointing 0.667 | energy 0.235 | energy ratio 7.3
  IG convergence: median delta 1.0% of the score change, 1 images needed more steps, 0 still above 5%
  chance level: pointing = tumor area share (about 0.02), energy ratio = 1


## 6. DenseNet121 (selected model): Grad-CAM + IG

In [6]:
!python -u -m src.xai.explain --model densenet121 --seed 42 --runs "{RUNS}" --out "{OUT}"

Explaining densenet121_lr0.0001_seed42 on cuda, methods ['gradcam', 'ig']

gradcam: 0 images already done, 860 to go -> /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/xai/densenet121_lr0.0001_seed42/gradcam_scores.csv
  25/860 images, 0.10s per image, about 1 min left
  50/860 images, 0.10s per image, about 1 min left
  75/860 images, 0.10s per image, about 1 min left
  100/860 images, 0.10s per image, about 1 min left
  125/860 images, 0.10s per image, about 1 min left
  150/860 images, 0.09s per image, about 1 min left
  175/860 images, 0.09s per image, about 1 min left
  200/860 images, 0.09s per image, about 1 min left
  225/860 images, 0.10s per image, about 1 min left
  250/860 images, 0.09s per image, about 1 min left
  275/860 images, 0.09s per image, about 1 min left
  300/860 images, 0.09s per image, about 1 min left
  325/860 images, 0.09s per image, about 1 min left
  350/860 images, 0.09s per image, about 1 min left
  375/860 images, 0.09s per image, about 1 m

## 7. ConvNeXt-Tiny: Grad-CAM + IG

In [7]:
!python -u -m src.xai.explain --model convnext_tiny --seed 43 --runs "{RUNS}" --out "{OUT}"

Explaining convnext_tiny_lr0.0001_seed43 on cuda, methods ['gradcam', 'ig']

gradcam: 0 images already done, 860 to go -> /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/xai/convnext_tiny_lr0.0001_seed43/gradcam_scores.csv
  25/860 images, 0.13s per image, about 2 min left
  50/860 images, 0.09s per image, about 1 min left
  75/860 images, 0.08s per image, about 1 min left
  100/860 images, 0.07s per image, about 1 min left
  125/860 images, 0.07s per image, about 1 min left
  150/860 images, 0.07s per image, about 1 min left
  175/860 images, 0.06s per image, about 1 min left
  200/860 images, 0.06s per image, about 1 min left
  225/860 images, 0.06s per image, about 1 min left
  250/860 images, 0.06s per image, about 1 min left
  275/860 images, 0.06s per image, about 1 min left
  300/860 images, 0.06s per image, about 1 min left
  325/860 images, 0.06s per image, about 1 min left
  350/860 images, 0.06s per image, about 0 min left
  375/860 images, 0.06s per image, about

## 8. ResNet50: Grad-CAM + IG

In [8]:
!python -u -m src.xai.explain --model resnet50 --seed 43 --runs "{RUNS}" --out "{OUT}"

Explaining resnet50_lr0.0001_seed43 on cuda, methods ['gradcam', 'ig']

gradcam: 0 images already done, 860 to go -> /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/xai/resnet50_lr0.0001_seed43/gradcam_scores.csv
  25/860 images, 0.09s per image, about 1 min left
  50/860 images, 0.07s per image, about 1 min left
  75/860 images, 0.07s per image, about 1 min left
  100/860 images, 0.06s per image, about 1 min left
  125/860 images, 0.06s per image, about 1 min left
  150/860 images, 0.06s per image, about 1 min left
  175/860 images, 0.05s per image, about 1 min left
  200/860 images, 0.05s per image, about 1 min left
  225/860 images, 0.05s per image, about 1 min left
  250/860 images, 0.05s per image, about 1 min left
  275/860 images, 0.05s per image, about 0 min left
  300/860 images, 0.05s per image, about 0 min left
  325/860 images, 0.05s per image, about 0 min left
  350/860 images, 0.05s per image, about 0 min left
  375/860 images, 0.05s per image, about 0 min lef

## 9. EfficientNetV2-S: Grad-CAM + IG

In [9]:
!python -u -m src.xai.explain --model efficientnet_v2_s --seed 44 --runs "{RUNS}" --out "{OUT}"

Explaining efficientnet_v2_s_lr0.0001_seed44 on cuda, methods ['gradcam', 'ig']

gradcam: 0 images already done, 860 to go -> /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/xai/efficientnet_v2_s_lr0.0001_seed44/gradcam_scores.csv
  25/860 images, 0.15s per image, about 2 min left
  50/860 images, 0.13s per image, about 2 min left
  75/860 images, 0.12s per image, about 2 min left
  100/860 images, 0.11s per image, about 1 min left
  125/860 images, 0.11s per image, about 1 min left
  150/860 images, 0.11s per image, about 1 min left
  175/860 images, 0.11s per image, about 1 min left
  200/860 images, 0.11s per image, about 1 min left
  225/860 images, 0.11s per image, about 1 min left
  250/860 images, 0.11s per image, about 1 min left
  275/860 images, 0.11s per image, about 1 min left
  300/860 images, 0.11s per image, about 1 min left
  325/860 images, 0.11s per image, about 1 min left
  350/860 images, 0.11s per image, about 1 min left
  375/860 images, 0.10s per imag

## 10. Grad-CAM only for the other 8 runs (for RQ4)

In [ ]:
for model, rep_seed in REP_SEEDS.items():
    for seed in ALL_SEEDS:
        if seed != rep_seed:
            !python -u -m src.xai.explain --model {model} --seed {seed} --runs "{RUNS}" --out "{OUT}" --methods gradcam

Explaining densenet121_lr0.0001_seed43 on cuda, methods ['gradcam']

gradcam: 0 images already done, 860 to go -> /content/drive/MyDrive/COMP6936 - Advanced Machine Learning/xai/densenet121_lr0.0001_seed43/gradcam_scores.csv
  25/860 images, 0.11s per image, about 2 min left
  50/860 images, 0.11s per image, about 1 min left
  75/860 images, 0.11s per image, about 1 min left
  100/860 images, 0.10s per image, about 1 min left
  125/860 images, 0.10s per image, about 1 min left
  150/860 images, 0.10s per image, about 1 min left
  175/860 images, 0.10s per image, about 1 min left
  200/860 images, 0.10s per image, about 1 min left
  225/860 images, 0.10s per image, about 1 min left
  250/860 images, 0.10s per image, about 1 min left
  275/860 images, 0.10s per image, about 1 min left
  300/860 images, 0.10s per image, about 1 min left
  325/860 images, 0.09s per image, about 1 min left
  350/860 images, 0.09s per image, about 1 min left
  375/860 images, 0.10s per image, about 1 min lef

## 11. Overview of everything on Drive
Mean scores for the predicted class. Chance level: pointing = tumor area share (about 0.02), energy ratio = 1.

In [ ]:
import glob
import pandas as pd

frames = [pd.read_csv(f) for f in glob.glob(f"{OUT}/*/*_scores.csv")]
scores = pd.concat(frames)
scores = scores[scores["target"] == "pred"]
overview = scores.groupby(["model", "seed", "method"]).agg(
    n=("path", "count"), pointing=("pointing_hit", "mean"),
    energy=("energy", "mean"), energy_ratio=("energy_ratio", "mean"))
overview.round(3)

n  pointing  energy  energy_ratio
model             seed method                                        
convnext_tiny     42   gradcam    860     0.426   0.174        10.334
                  43   gradcam    860     0.428   0.170         9.969
                       ig_raw     860     0.776   0.202        13.732
                       ig_smooth  860     0.794   0.191        12.701
                  44   gradcam    860     0.344   0.164         9.628
densenet121       42   gradcam    860     0.499   0.095         4.706
                       ig_raw     860     0.579   0.143         8.464
                       ig_smooth  860     0.633   0.138         8.008
                  43   gradcam    860     0.407   0.093         4.494
                  44   gradcam    860     0.512   0.095         4.689
efficientnet_v2_s 42   gradcam    860     0.166   0.081         4.061
                  43   gradcam    860     0.150   0.087         4.373
                  44   gradcam    860     0.114   0.074         3.902
                       ig_raw     860     0.584   0.142         8.400
                       ig_smooth  860     0.603   0.137         7.931
resnet50          42   gradcam    860     0.400   0.158         8.377
                  43   gradcam    860     0.386   0.164         8.383
                       ig_raw     860     0.541   0.141         8.234
                       ig_smooth  860     0.626   0.136         7.844
                  44   gradcam    860     0.416   0.161         7.962

## 12. After everything has run
- Download the `xai` folder from Drive (`COMP6936 - Advanced Machine Learning/xai/`) into the repo as `results/xai/`.
- The `*_scores.csv` files get committed. The `maps/` folders are gitignored (example heatmaps for figures only).